# LogisChain AI — 03: Model Training

Train and evaluate all five model families: GNN, TCN, Transformer, XGBoost/LightGBM, Survival.

**Goals:**
- Train XGBoost and LightGBM on carrier default prediction
- Train CarrierSurvivalModel for time-to-failure
- Train LogisChainEnsemble via stacking
- Log all experiments to MLflow
- Compare model performance

In [ ]:
import sys; sys.path.insert(0, '..')
import pandas as pd
import numpy as np
import mlflow
import mlflow.sklearn
from src.data.pipeline import SyntheticDataGenerator
from src.features.fusion_features import FeaturePipeline
from src.models.xgboost_model import XGBoostRiskModel, LightGBMRiskModel
from src.models.survival import CarrierSurvivalModel
from src.models.ensemble import LogisChainEnsemble
from src.utils.metrics import classification_report_dict

mlflow.set_experiment('logischain_ai')
gen = SyntheticDataGenerator(seed=42)
data = gen.generate_all()
print('Data ready.')

In [ ]:
# Prepare features
fp = FeaturePipeline()
fused = fp.run(data['carriers'], data['shipments'], data['financial'])
target_col = 'carrier_failure'
if target_col in fused.columns:
    y = fused[target_col].fillna(0)
    drop_cols = [target_col, 'carrier_id', 'company_id', 'carrier_type', 'region', 'industry']
    X = fused.drop(columns=[c for c in drop_cols if c in fused.columns]).select_dtypes(include=np.number).fillna(0)
    from sklearn.model_selection import train_test_split
    X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, stratify=y, random_state=42)
    print(f'Train: {X_train.shape}, Test: {X_test.shape}')

In [ ]:
# Train XGBoost
with mlflow.start_run(run_name='xgboost_carrier_default'):
    xgb_model = XGBoostRiskModel(config={'n_estimators': 300, 'learning_rate': 0.02})
    xgb_model.fit(X_train, y_train, X_test, y_test)
    xgb_probs = xgb_model.predict_proba(X_test)
    xgb_metrics = classification_report_dict(y_test.values, xgb_probs)
    mlflow.log_metrics({k: v for k, v in xgb_metrics.items() if isinstance(v, float)})
    print('XGBoost:', {k: round(v, 4) for k, v in xgb_metrics.items() if k in ['roc_auc', 'f1', 'ks_statistic']})

In [ ]:
# Train LightGBM
with mlflow.start_run(run_name='lightgbm_carrier_default'):
    lgb_model = LightGBMRiskModel(config={'n_estimators': 300})
    lgb_model.fit(X_train, y_train, X_test, y_test)
    lgb_probs = lgb_model.predict_proba(X_test)
    lgb_metrics = classification_report_dict(y_test.values, lgb_probs)
    mlflow.log_metrics({k: v for k, v in lgb_metrics.items() if isinstance(v, float)})
    print('LightGBM:', {k: round(v, 4) for k, v in lgb_metrics.items() if k in ['roc_auc', 'f1', 'ks_statistic']})

In [ ]:
# Train Ensemble
pred_dict = {'xgboost': xgb_probs, 'lightgbm': lgb_probs}
train_preds = {'xgboost': xgb_model.predict_proba(X_train), 'lightgbm': lgb_model.predict_proba(X_train)}
ensemble = LogisChainEnsemble()
ensemble.fit_from_predictions(train_preds, y_train.values)
ens_metrics = ensemble.evaluate(pred_dict, y_test.values)
print('Ensemble:', {k: round(v, 4) for k, v in ens_metrics.items()})